## **Transformers Walkthrough**
This script is a walk-through of the Transformers architecture. We demonstrate the various Transformers component in a language translation task.

## Load pre-trained model and tokenizer

In [1]:
# Not needed if your kernel already has these packages; this is for my local Jupyter instance
#Cell can be deleted

!uv add "transformers==4.46.3" "torch==2.2.2" "tokenizers==0.20.3" sentencepiece protobuf

Resolved 89 packages in 14ms
Checked 68 packages in 42ms


In [2]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import torch

checkpoint = 'google/mt5-small'
tokenizer = AutoTokenizer.from_pretrained(checkpoint,
                                          additional_special_tokens=['<2ceb>','<2es>','<2en>','<2hil>','<2cbk>']
                                          )
model = AutoModelForSeq2SeqLM.from_pretrained(checkpoint)

device = torch.device("mps" if torch.backends.mps.is_available()
                      else "cuda" if torch.cuda.is_available()
                      else "cpu")
model = model.to(device)

/Users/normanjazul/Documents/vscode/Python/Notebooks/NLP/EXERCISES/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm

A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.5.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/Users/normanjazul/Documents/vscode/Python/Notebooks/NLP/EXERCISES/.venv/lib/python3.12/site-packages/ipyker

In [ ]:
model.config

In [ ]:
'''
ANSWER.
Explore the mt5 vocabulary using get_vocab().
'''

vocab = tokenizer.get_vocab()

In [ ]:
'''
Question 1. What is the size of the mT5 vocabulary?
The actual vocabulary size is less than 5 since we added some special tokens (language tags)
'''

len(vocab)

In [ ]:
'''
Question 2. What is the structure of the mT5 vocabulary?
'''
vocab

In [ ]:
'''
Question 3. What is the token ID for the special token '<pad>'?
Remember that this value may differ in other models.
'''
vocab['<pad>']

## Explore Input Embedding
This section is not really a part of model training but I just wanted to show you how Transformers handle token embeddings.

In [ ]:
input_text = "The black cat sat on the couch and the brown dog slept on the rug."
inputs = tokenizer(input_text, return_tensors="pt")
input_ids = inputs["input_ids"]
input_tokens = tokenizer.convert_ids_to_tokens(input_ids.flatten())
print(input_ids)
print(input_tokens)

In [ ]:
# Extract the input embeddings directly
embedding_layer = model.shared  # nn.Embedding; embedding matrix shared between the encoder and decoder
embedding_layer.to('cpu') # I have to force cpu computing because pytorch have issues with m3
input_embeddings = embedding_layer(input_ids).to('cpu')  # shape: [batch_size, seq_len, hidden_size]

print(input_embeddings.shape) 

In [ ]:
'''
Cosine similarity measures the cosine of the angle between two vectors in a high-dimensional space.
+1 → Vectors point in the same direction → Maximum similarity
 0 → Vectors are orthogonal (unrelated) → No similarity
−1 → Vectors point in opposite directions → Maximum dissimilarity
'''
cosine_similarity = torch.nn.CosineSimilarity(dim=0, eps=1e-8)
print(f'similarity between {input_tokens[1]} and {input_tokens[2]}: {cosine_similarity(input_embeddings[0][1],input_embeddings[0][2])}')
print(f'similarity between {input_tokens[1]} and {input_tokens[10]}: {cosine_similarity(input_embeddings[0][1],input_embeddings[0][10])}')
print(f'similarity between {input_tokens[2]} and {input_tokens[11]}: {cosine_similarity(input_embeddings[0][2],input_embeddings[0][11])}')

## Tokenization

In [ ]:
from datasets import load_from_disk
dataset_dir = '/Users/aileenjoanvicente/lexmark/exercises/small_dataset'
dataset = load_from_disk(dataset_dir)

In [ ]:
'''
Question 4. Describe the dataset in terms of how the samples are structured.
'''
dataset

In [ ]:
'''
Prepare inputs and targets as lists.
'''
def preprocess_function(examples):
    inputs = [str(example) for example in examples['source']]
    targets = [str(example) for example in examples['target']]
    return tokenizer(inputs, text_target=targets, max_length=128 ,truncation=True, padding='max_length', return_tensors="pt")


'''
Map() applies a function to each example or batch in your dataset to transform or tokenize it, typically before training.
This function can create new rows and columns or add prefixes.
'''
tokenized_dataset = dataset.map(preprocess_function, batched=True)

In [ ]:
tokenized_dataset

In [ ]:
#load tokenized dataset
from datasets import load_from_disk
tokenized_dir = '/Users/aileenjoanvicente/lexmark/exercises/m2n_tokenized_small'
tokenized_dataset = load_from_disk(tokenized_dir)

In [ ]:
'''
Explore the tokenized dataset.
Question 5: Describe the tokenized dataset in terms of how the samples are structured.
'''
tokenized_dataset

In [ ]:
'''
Retrieve the first input sample of the tokenized training dataset and convert this back to tokens using
tokenizer.convert_ids_to_tokens()
Question 6: Describe the tokens in the tokenized dataset.
'''
print(tokenizer.convert_ids_to_tokens(tokenized_dataset['train']['input_ids'][0]))
print(dataset['train']['source'][0])


## Finetuning
Since we only have few training data, it is better (in practice) to fine-tune a pre-trained model. 
If you decide to train from scratch using the configuration of a model, you may do so by creating an instance of the model without loading the weights. Here's an example:

    # Load mt5 configuration
      vocab_size = 30000
      config = MT5Config.from_pretrained('google/mt5-base',vocab_size=vocab_size)

    # Create a model from an existing configuration, does not load pre-trained weights
      model = TFMT5ForConditionalGeneration.from_config(config)
      optimizer = AdamWeightDecay(learning_rate=0.001)

      model.compile(optimizer=optimizer)

You can also train your own tokenizer and build your own vocabulary. Please check the Huggingface documentation on how to do these.

In [ ]:
# view the model's configuration
# mT5 checkpoints vary in  the model ffn size
model.config

In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq
import torch

# Define training arguments
training_args = TrainingArguments(
    output_dir="/Users/aileenjoanvicente/lexmark/exercises/outputs",
    per_device_train_batch_size=8,  #Trainer handles batching!
    per_device_eval_batch_size=8,
    learning_rate=2e-5,             #The learning rate controls how big a step the optimizer takes when updating the model’s parameters (weights) during training.
    weight_decay=0.01,              #Regularization; penalizes large weights in the model by adding a term to the loss function that grows with the size of the weights.
    num_train_epochs=3,
    eval_strategy="epoch",          #1 epoch = 1 full pass through the entire training dataset
    torch_empty_cache_steps = 2,    #1 step = 1 forward pass + 1 backward pass + 1 weight update; if batch_size =  64, then 1 step = processing 64 training examples
    save_strategy = 'epoch',        #for saving checkpoints
    logging_strategy="steps",       #to log training loss
    logging_steps=10
)

#Data collator is used at runtime by the DataLoader to collate and pad a batch of already-tokenized examples
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, return_tensors="pt", label_pad_token_id = 0, padding=True)

# Define Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['valid'],
    # Include data_collator to format the data correctly for MT5
    data_collator= data_collator 
)

#Start training
trainer.train()

## Evaluating Model using BLEU

In [ ]:
tokenized_dataset['test']

In [ ]:
import evaluate
from tqdm import tqdm


# Set device (MPS for Mac, CUDA for NVIDIA, CPU fallback)
#device = torch.device("mps" if torch.backends.mps.is_available()
#                      else "cuda" if torch.cuda.is_available()
#                      else "cpu")
#model = model.to(device)
model.eval() # use this when evaluating the model

# Load BLEU metric
bleu = evaluate.load("sacrebleu")

# Define test inputs and references
# Example structure:
# tokenized_dataset['test'] = {'input_ids': [...], 'attention_mask': [...], 'labels_text': [...]}

batch_size = 16  # adjust depending on available memory
all_predictions = []
all_references = []

# Split test set into batches
n = len(tokenized_dataset['test']['input_ids'])
for i in tqdm(range(0, n, batch_size)):
    input_ids = torch.tensor(tokenized_dataset['test']['input_ids'][i:i+batch_size]).to(device)
    attention_mask = torch.tensor(tokenized_dataset['test']['attention_mask'][i:i+batch_size]).to(device)

    with torch.no_grad(): #disable gradient calculation since we are evaluating the model
        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_length=64,
            num_beams=4
        )

    decoded_preds = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    all_predictions.extend(decoded_preds)

    batch_refs = tokenized_dataset['test']['target'][i:i+batch_size]
    all_references.extend([[ref] for ref in batch_refs])  # BLEU expects list of lists

# Compute BLEU score
bleu_result = bleu.compute(predictions=all_predictions, references=all_references)
print(f"\nBLEU Score: {bleu_result['score']:.2f}")


In [ ]:
tokenized_dataset['test']['input_ids'][0]

In [ ]:
# Generate translations
model_inputs = tokenizer(inputs, return_tensors="pt", padding=True, truncation=True).to(device)

with torch.no_grad():
    outputs = model.generate(
        input_ids=model_inputs['input_ids'],
        attention_mask=model_inputs['attention_mask'],
        max_length=64
    )

preds = tokenizer.batch_decode(outputs, skip_special_tokens=True)

# Compute BLEU score
bleu = load_metric("sacrebleu")
bleu_result = bleu.compute(predictions=preds, references=[[ref] for ref in references])

print(f"\nBLEU Score: {bleu_result['score']:.2f}")